# TicketAI - Customer Support Ticket Classification Pipeline
### Production Machine Learning Pipeline with High-Accuracy Multi-Class Classification

This notebook builds, benchmarks, and exports a production-grade NLP classifier for customer support tickets into 5 core operational categories:
1. **Technical Support**: Login issues, server errors, bugs, app crashes, connectivity.
2. **Billing**: Invoices, duplicate charges, payment failures, subscription renewals.
3. **Returns**: Return requests, damaged goods, wrong items received, refunds.
4. **General Inquiry**: Store hours, product specifications, shipping duration, company info.
5. **Complaint**: Service dissatisfaction, rude staff, delivery delays, escalations.


In [1]:
# Step 1: Import required libraries
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score

for res in ["stopwords", "wordnet"]:
    nltk.download(res, quiet=True)
print("Libraries loaded successfully!")


Libraries loaded successfully!


## 2. Dataset Loading & Preprocessing

We load our comprehensive, balanced dataset representing realistic customer inquiries with distinct linguistic patterns and vocabulary across all 5 operational categories.


In [2]:
# Step 2: Load training dataset and initialize text preprocessor
from train_model import get_comprehensive_dataset, ProductionTextPreprocessor

df = get_comprehensive_dataset()
print(f"Total training records: {len(df)}")
print("\nClass distribution:")
print(df["category"].value_counts())

preprocessor = ProductionTextPreprocessor()
df["cleaned_text"] = df["text"].apply(preprocessor.preprocess)
print("\nSample processed text:")
print(df[["text", "cleaned_text", "category"]].head(3))


[DATA] Ingesting CSV dataset from: c:\ai_ticket\models\customer_support_tickets.csv
[DATA] Ingested 8469 records from CSV with conversational paraphrasing.
[DATA] Hardened dataset size: 10005 total records across 5 categories.
Total training records: 10005

Class distribution:
category
Technical Support    5369
General Inquiry      1832
Complaint            1216
Billing               842
Returns               746
Name: count, dtype: int64

Sample processed text:
                                                text  \
0  Need guidance setting up product: I'm having a...   
1  Connecting accessory question: I'm having an i...   
2  Cannot reach network: I'm facing a problem wit...   

                                        cleaned_text           category  
0  need guidance setting product issue gopro hero...  Technical Support  
1  connecting accessory question issue lg smart t...    General Inquiry  
2  cannot reach network facing problem dell xps d...  Technical Support  


## 3. Feature Extraction & Multi-Model Benchmarking

We construct a dual-granularity feature union:
- **Word-level TF-IDF (1-2 ngrams)**: Captures domain phrases and context.
- **Character-level TF-IDF (3-5 ngrams)**: Captures sub-word roots, typos, and variations.

We benchmark candidate algorithms using 5-Fold Stratified Cross Validation.


In [3]:
# Step 3: Train / Test Split and Multi-Model Benchmarking
X = df["cleaned_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

feature_union = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=4000)),
    ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, max_features=4000))
])

models = {
    "Calibrated LinearSVC": CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, class_weight="balanced"), cv=3),
    "Logistic Regression": LogisticRegression(C=2.0, max_iter=1000, class_weight="balanced", random_state=42),
    "Complement Naive Bayes": ComplementNB(alpha=0.5)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Cross-Validation Benchmark Results:")
for name, clf in models.items():
    pipe = Pipeline([("features", feature_union), ("clf", clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy")
    pipe.fit(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    print(f"• {name:<26} | CV Acc: {scores.mean()*100:.1f}% | Test Acc: {test_acc*100:.1f}%")


Cross-Validation Benchmark Results:
• Calibrated LinearSVC       | CV Acc: 98.4% | Test Acc: 98.9%
• Logistic Regression        | CV Acc: 98.2% | Test Acc: 98.4%
• Complement Naive Bayes     | CV Acc: 95.6% | Test Acc: 96.6%


## 4. Evaluation & Production Model Verification

We load the saved production model and verify its predictions on realistic customer support tickets.


In [4]:
# Step 4: Verify Saved Production Model
model_path = r"C:\ai_ticket\models\enhanced_ticket_classifier.joblib"
production_pipeline = joblib.load(model_path)

test_samples = [
    "My database connection is failing with 500 error code",
    "I was charged twice on my credit card for invoice #4820",
    "The product arrived broken with shattered glass, want a refund",
    "What are your store hours and what is your phone number?",
    "Terrible service! Waiting for 3 weeks and agent was extremely rude!"
]

print("--- Real-World Inference Verification ---")
for text in test_samples:
    clean = preprocessor.preprocess(text)
    probs = production_pipeline.predict_proba([clean])[0]
    idx = np.argmax(probs)
    pred = production_pipeline.classes_[idx]
    conf = probs[idx] * 100
    print(f"Input: \"{text}\"")
    print(f"  -> Category: {pred} ({conf:.1f}% confidence)\n")


--- Real-World Inference Verification ---
Input: "My database connection is failing with 500 error code"
  -> Category: Technical Support (97.6% confidence)

Input: "I was charged twice on my credit card for invoice #4820"
  -> Category: Billing (95.8% confidence)

Input: "The product arrived broken with shattered glass, want a refund"
  -> Category: Returns (67.7% confidence)

Input: "What are your store hours and what is your phone number?"
  -> Category: General Inquiry (70.8% confidence)

Input: "Terrible service! Waiting for 3 weeks and agent was extremely rude!"
  -> Category: Complaint (72.1% confidence)

